In [1]:
import sqlite3
import pandas as pd
import time
from datetime import datetime
from functools import wraps
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

from crewai.tools import tool
from crewai import Agent, Task, Crew, Process
from crewai import LLM
#from agentics import AG

# ============================================
# Global Walk-Forward State
# ============================================
PIPELINE_STATE = {
    "current_eval_date": None,  
    "allow_returns": True       
}

# ============================================
# Rate Limiting & Tools Setup
# ============================================
class RateLimiter:
    def __init__(self, calls_per_minute=5):
        self.interval = 60.0 / calls_per_minute
        self.last_call = 0
        
    def wait_if_needed(self):
        current_time = time.time()
        time_since_last = current_time - self.last_call
        if time_since_last < self.interval:
            time.sleep(self.interval - time_since_last)
        self.last_call = time.time()

rate_limiter = RateLimiter(calls_per_minute=5)

def rate_limit(func):
    @wraps(func)
    @retry(
        retry=retry_if_exception_type((Exception,)),
        wait=wait_exponential(multiplier=1, min=4, max=60),
        stop=stop_after_attempt(5)
    )
    def wrapper(*args, **kwargs):
        rate_limiter.wait_if_needed()
        try:
            return func(*args, **kwargs)
        except Exception as e:
            raise
    return wrapper

import sqlite3
import pandas as pd

def setup_firm_database(csv_path, db_name='firm_data.db'):
    print(f"Loading fundamental data from {csv_path}...")
    
    # Load your fundamentals CSV (e.g., 'green cleaned.csv')
    df = pd.read_csv(csv_path)
    
    # 1. Ensure required columns exist
    required_cols = ['datadate', 'ticker', 'mve', 'bm', 'mom12m', 'ret_fwd_1']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required fundamental column: {col}")
            
    # 2. Format dates to strict YYYY-MM-DD strings so SQLite BETWEEN queries work
    df['datadate'] = pd.to_datetime(df['datadate']).dt.strftime('%Y-%m-%d')
    
    # 3. Create SQLite Database
    print(f"Creating {db_name}...")
    conn = sqlite3.connect(db_name)
    
    # Write to table named 'firms' (required by your tool)
    df.to_sql('firms', conn, if_exists='replace', index=False)
    
    # 4. Create Indexes for faster querying
    conn.execute("CREATE INDEX IF NOT EXISTS idx_firm_date ON firms(datadate)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_firm_ticker ON firms(ticker)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_firm_date_ticker ON firms(datadate, ticker)")
    conn.close()
    print("✓ firm_data.db created successfully.")

def setup_sentiment_database(csv_path, db_name='sentiment_data.db'):
    print(f"\nLoading RavenPack data from {csv_path}...")
    
    # Load your daily RavenPack CSV
    df = pd.read_csv(csv_path)
    
    # 1. Ensure required columns exist
    required_cols = ['date', 'ticker', 'clean_headline', 'event_sentiment_score', 'ret_fwd_1']
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required sentiment column: {col}")
            
    # 2. Format dates to strict YYYY-MM-DD
    df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')
    
    # 3. Create SQLite Database
    print(f"Creating {db_name}...")
    conn = sqlite3.connect(db_name)
    
    # Write to table named 'ravenpack' (required by your tool)
    df.to_sql('ravenpack', conn, if_exists='replace', index=False)
    
    # 4. Create Indexes (CRITICAL for the 30-day trailing aggregation queries)
    conn.execute("CREATE INDEX IF NOT EXISTS idx_rp_date ON ravenpack(date)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_rp_ticker ON ravenpack(ticker)")
    conn.execute("CREATE INDEX IF NOT EXISTS idx_rp_date_ticker ON ravenpack(date, ticker)")
    
    conn.close()
    print("✓ sentiment_data.db created successfully.")

# --- Fundamental Tools ---
@tool("query_firm_database")
@rate_limit
def query_firm_database(sql_query: str) -> str:
    """Execute SQL queries on the firms database."""
    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sql_query.lower():
        return f"SECURITY BLOCK: Access to 'ret_fwd_1' is strictly prohibited during evaluation."
    
    try:
        conn = sqlite3.connect('firm_data.db')
        df = pd.read_sql_query(sql_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in df.columns:
            df = df.drop(columns=["ret_fwd_1"])
            
        if df.empty: return "Query returned no results."
        return f"Query returned {len(df)} rows:\n\n" + df.head(5).to_string(index=False)
    except Exception as e:
        return f"Error executing query: {str(e)}"

@tool("get_database_schema")
@rate_limit
def get_database_schema() -> str:
    """Get the schema of the firms database."""
    try:
        conn = sqlite3.connect('firm_data.db')
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(firms)")
        columns = cursor.fetchall()
        conn.close()
        
        schema = "Database: firm_data.db\nTable: firms\nColumns:\n"
        for col in columns:
            if not PIPELINE_STATE["allow_returns"] and col[1] == "ret_fwd_1":
                continue 
            schema += f"  - {col[1]} ({col[2]})\n"
        return schema
    except Exception as e:
        return f"Error retrieving schema: {str(e)}"


@tool("get_sentiment_schema")
@rate_limit
def get_sentiment_schema() -> str:
    """Get the schema of the sentiment database."""
    try:
        conn = sqlite3.connect('sentiment_data.db')
        cursor = conn.cursor()
        cursor.execute("PRAGMA table_info(ravenpack)")
        columns = cursor.fetchall()
        conn.close()
        
        schema = "Database: sentiment_data.db\nTable: ravenpack\nColumns:\n"
        for col in columns:
            if not PIPELINE_STATE["allow_returns"] and col[1] == "ret_fwd_1":
                continue 
            schema += f"  - {col[1]} ({col[2]})\n"
        return schema
    except Exception as e:
        return f"Error retrieving schema: {str(e)}"

@tool("test_complex_condition")
@rate_limit
def test_complex_condition(sql_condition: str, date: str = None) -> str:
    """Test complex conditions with AND/OR/NOT logic on fundamentals."""
    if "WHERE " in sql_condition.upper():
        sql_condition = sql_condition[sql_condition.upper().find("WHERE ") + 6:]
    elif "SELECT " in sql_condition.upper():
        return "Error: Pass ONLY the condition string. Do not pass a SELECT query."

    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sql_condition.lower():
        return f"SECURITY BLOCK: You cannot use 'ret_fwd_1' during zero-shot evaluation."
        
    try:
        conn = sqlite3.connect('firm_data.db')
        target_date = date if date else PIPELINE_STATE["current_eval_date"]
        
        if " to " in target_date:
            start_date, end_date = target_date.split(" to ")
            date_clause = f"datadate BETWEEN '{start_date.strip()}' AND '{end_date.strip()}'"
        else:
            date_clause = f"datadate = '{target_date.strip()}'"
        
        count_query = f"SELECT COUNT(*) as n_firms FROM firms WHERE {date_clause} AND ({sql_condition})"
        columns_to_select = "datadate, ticker, mve, bm, mom12m" if 'ticker' in sql_condition else "*"
        sample_query = f"SELECT {columns_to_select} FROM firms WHERE {date_clause} AND ({sql_condition}) LIMIT 5"
        
        n_firms = pd.read_sql_query(count_query, conn)['n_firms'].values[0]
        sample_df = pd.read_sql_query(sample_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sample_df.columns:
            sample_df = sample_df.drop(columns=["ret_fwd_1"])
            
        result = f"Condition Test at {target_date}:\nFirms meeting condition: {n_firms:,}\n\n"
        return result + (sample_df.to_string(index=False) if not sample_df.empty else "(No firms meet this condition)")
    except Exception as e:
        return f"Error testing condition: {str(e)}"

# --- Sentiment Tools ---
@tool("query_sentiment_database")
@rate_limit
def query_sentiment_database(sql_query: str) -> str:
    """Execute SQL queries on the daily RavenPack sentiment database."""
    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sql_query.lower():
        return f"SECURITY BLOCK: Access to future returns is prohibited."
    
    try:
        conn = sqlite3.connect('sentiment_data.db')
        df = pd.read_sql_query(sql_query, conn)
        conn.close()
        
        if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in df.columns:
            df = df.drop(columns=["ret_fwd_1"])
            
        if df.empty: return "Query returned no results."
        return f"Query returned {len(df)} rows:\n\n" + df.head(5).to_string(index=False)
    except Exception as e:
        return f"Error executing query: {str(e)}"

@tool("test_sentiment_condition")
@rate_limit
def test_sentiment_condition(sql_condition: str, eval_date: str = None) -> str:
    """
    Test sentiment conditions on the RavenPack dataset. 
    Pass ONLY the condition (e.g., 'avg_sentiment > 0.5 AND news_count > 5').
    Automatically aggregates the trailing 30 days prior to the eval_date.
    """
    if "WHERE " in sql_condition.upper():
        sql_condition = sql_condition[sql_condition.upper().find("WHERE ") + 6:]

    # Block access to ret_fwd_1 during zero-shot eval, just like the fundamental tool
    if not PIPELINE_STATE["allow_returns"] and "ret_fwd_1" in sql_condition.lower():
        return f"SECURITY BLOCK: You cannot use 'ret_fwd_1' during zero-shot evaluation."

    try:
        conn = sqlite3.connect('sentiment_data.db')
        target_date = eval_date if eval_date else PIPELINE_STATE["current_eval_date"]
        
        end_dt = pd.to_datetime(target_date)
        start_dt = end_dt - pd.Timedelta(days=30)
        
        # Updated to use 'date' and 'event_sentiment_score'
        query = f"""
        SELECT ticker, AVG(event_sentiment_score) as avg_sentiment, COUNT(clean_headline) as news_count 
        FROM ravenpack 
        WHERE date BETWEEN '{start_dt.strftime('%Y-%m-%d')}' AND '{end_dt.strftime('%Y-%m-%d')}'
        GROUP BY ticker
        HAVING ({sql_condition})
        """
        
        df = pd.read_sql_query(query, conn)
        conn.close()
        
        result = f"Sentiment Condition Test (Trailing 30 days ending {target_date}):\nFirms meeting condition: {len(df):,}\n\n"
        return result + (df.head(5).to_string(index=False) if not df.empty else "(No firms meet this condition)")
    except Exception as e:
        return f"Error testing sentiment condition: {str(e)}"


def get_rate_limited_llm():
    """Get Local Ollama LLM using CrewAI's native LiteLLM wrapper."""
    llm = LLM(
        model="ollama/llama3", # Explicitly requesting llama3, not llama3.1
        base_url="http://localhost:11434", # Default local Ollama port
        timeout=720, # Keep this high; local models need time to process complex SQL tasks
        temperature=0.0  # <--- Set temperature to absolute minimum
    )
    return llm

# ============================================
# INSTRUCTIONS & BACKSTORIES
# ============================================
QUANT_BACKSTORY = """
You are an expert quantitative strategist who creates systematic, rule-based trading strategies.
    
CRITICAL DATA UNDERSTANDING:
- 'mve' = log(market value of equity), represents log firm size
- 'bm' = book-to-market ratio (value factor). High bm = undervalued, low bm = overvalued.
- 'mom12m' = 12-month momentum
- ALL features are standardized z-scores (mean=0, std=1). Avoid z-scores > 3 or < -3 to prevent 0-firm portfolios.

Your task is to develop EXPLICIT, SYSTEMATIC RULES for generating fundamental trading signals based on the provided data. Test your conditions using test_complex_condition.
"""

SENTIMENT_BACKSTORY = """
You are an expert alternative data quantitative strategist specializing in NLP and news sentiment.

CRITICAL DATA UNDERSTANDING:
- You have access to daily RavenPack news data, containing 'ticker', 'date', 'clean_headline', 'event_sentiment_score', and 'ret_fwd_1'.
- You MUST query 'ret_fwd_1' during the training phase to evaluate if your signals accurately predict forward returns.
- Since our strategy rebalances MONTHLY, daily noise is a liability. You must construct rules based on trailing 30-day aggregations (e.g., 'avg_sentiment', 'news_count').
- A high sentiment score on low news volume (news_count = 1) is often statistical noise. A high sentiment score on high volume indicates strong consensus.

Your task is to develop EXPLICIT, SYSTEMATIC RULES for generating trading signals based PURELY on trailing 30-day sentiment behavior. Test your conditions using test_sentiment_condition.
"""

QUANT_EXPECTED_OUTPUT = """
Complete strategy document with:
1. Data exploration summary
2. Explicit BUY rule with z-score thresholds
3. Explicit SELL rule with z-score thresholds  
4. HOLD rule (default case)
5. Economic rationale for each rule

OUTPUT FORMAT:
===========================================
SYSTEMATIC TRADING RULES
===========================================

Data Exploration Summary:
- [Key statistics and patterns observed]

BUY RULE:
if [ANY complex z-score condition using AND/OR/NOT]:
    signal = BUY

Examples of valid BUY rules:
- "bm > 1.15 AND mom12m > 0.73 AND mve < 1.11" (simple AND)
- "bm > 1.56 OR mom12m > 1.28 OR mve > 1.52" (simple OR)
- "(bm > 1.02 AND mom12m > 0.53) OR mve > 1.59" (combination)
- "bm > 0.83 AND (mom12m > 0.77 OR mve > 1.08)" (nested conditions)

SELL RULE:
if [ANY complex z-score condition]:
    signal = SELL

HOLD RULE:
else:
    signal = HOLD

Rationale:
- [Economic reasoning for BUY rule]
- [Economic reasoning for SELL rule]
- [Expected signal distribution]

===========================================

Be precise, systematic, and data-driven. Your rules will be applied to ~500 firms.
"""


SENTIMENT_EXPECTED_OUTPUT = """
Complete strategy document with:
1. Data exploration summary (focusing on 30-day trailing averages and news volume)
2. Explicit BUY rule with thresholds
3. Explicit SELL rule with thresholds  
4. HOLD rule (default case)
5. Economic rationale for each rule

OUTPUT FORMAT:
===========================================
SYSTEMATIC TRADING RULES (SENTIMENT)
===========================================

Data Exploration Summary:
- [Key statistics and patterns observed in 30-day sentiment and volume]

BUY RULE:
if [ANY complex condition using avg_sentiment and news_count]:
    signal = BUY

Examples of valid BUY rules:
- "avg_sentiment > 0.85 AND news_count >= 5" (high sentiment, high consensus)
- "avg_sentiment > 0.60 AND (news_count > 10 OR avg_sentiment > 0.90)"

SELL RULE:
if [ANY complex condition using avg_sentiment and news_count]:
    signal = SELL

HOLD RULE:
else:
    signal = HOLD

Rationale:
- [Reasoning for BUY rule based on NLP/sentiment logic]
- [Reasoning for SELL rule based on NLP/sentiment logic]

===========================================

Be precise and data-driven. Do NOT use z-scores. Use raw 'avg_sentiment' and 'news_count' values.
"""

INTEGRATED_EXPECTED_OUTPUT = """
Complete integrated strategy document combining Fundamental and Sentiment signals.

OUTPUT FORMAT:
===========================================
FINAL INTEGRATED TRADING STRATEGY
===========================================

1. FUNDAMENTAL RULES (mve, bm, mom12m as z-scores):
- BUY if: [exact condition]
- SELL if: [exact condition]

2. SENTIMENT RULES (30-day trailing avg_sentiment, news_count):
- BUY if: [exact condition]
- SELL if: [exact condition]

3. UNIFIED BOOLEAN INTEGRATION RULE (Final Portfolio Selection):
- You MUST combine the fundamental and sentiment variables into a SINGLE explicit boolean equation.
- DO NOT use a point-based scoring system. 
- DO NOT use textual descriptions like "take the intersection". 
- FORMAT EXAMPLE: 
  BUY if: (bm > 1.15 AND mom12m > 0.5) AND (avg_sentiment > 0.75 AND news_count >= 3)
  SELL if: (bm < -1.0) OR (avg_sentiment < 0.2 AND news_count >= 5)

4. ECONOMIC RATIONALE:
- [Why this combined boolean strategy captures the best cross-domain edge]
===========================================
"""

# ============================================
# ACE Agents
# ============================================
fundamentals_generator = Agent(
    role="Fundamental Quant Developer",
    goal="Develop structural rules using mve, bm, and mom12m, test against 'ret_fwd_1', and record reasoning.",
    backstory=QUANT_BACKSTORY,
    tools=[get_database_schema, query_firm_database, test_complex_condition],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5,
    max_rpm=10
)

sentiment_generator = Agent(
    role="Alternative Data Quant Developer",
    goal="Develop sentiment trading rules based on 30-day trailing averages of RavenPack data.",
    backstory=SENTIMENT_BACKSTORY,
    tools=[get_sentiment_schema, query_sentiment_database, test_sentiment_condition], 
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5,
    max_rpm=10
)

reflector_agent = Agent(
    role="Strategy Reflector",
    goal="Analyze execution logs from BOTH fundamental and sentiment generators and formulate a unified boolean rule.",
    backstory="""You are an expert portfolio manager. Review both the Generator's reasoning and the actual realized outcomes ('ret_fwd_1'). 
    Identify prediction failures. You must integrate fundamental and sentiment variables into a SINGLE explicit boolean condition (e.g., combining bm, mom12m, avg_sentiment, and news_count with AND/OR logic). 
    NEVER use arbitrary point-based scoring systems or vague textual rules like 'take the intersection'.""",
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5,
    max_rpm=10
)

curator_agent = Agent(
    role="Rulebook Curator",
    goal="Integrate delta updates into a cohesive master rulebook featuring a unified cross-domain boolean rule.",
    backstory="""You manage the master rulebook. Integrate proposed deltas from the Reflector. 
    Your most critical job is to output the final Portfolio Signal as a STRICT, unified boolean mathematical equation containing BOTH fundamental features and sentiment features in the same 'IF' statement. Do not use points or scores.""",
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5,
    max_rpm=10
)

evaluation_agent = Agent(
    role="Final Strategy Evaluator",
    goal="Execute the optimized rulebook on out-of-sample data to generate final trading signals.",
    backstory="""You must use causal masking to prevent any look-ahead bias. Take the finalized ACE strategy rulebook and apply it strictly to the current evaluation date using BOTH fundamental and sentiment tools. 
    You DO NOT have access to future returns. Output explicit thresholds so they can be implemented in production.""",
    tools=[
        get_database_schema, query_firm_database, test_complex_condition,
        get_sentiment_schema, query_sentiment_database, test_sentiment_condition
    ],
    llm=get_rate_limited_llm(),
    verbose=True,
    max_iter=5,
    max_rpm=10
)

# ============================================
# Core Functions
# ============================================
def run_ace_loop(training_start, training_end, iterations=3):
    current_rulebook = """BASE RULEBOOK v1.0: 
[FUNDAMENTALS]: Find z-score combinations for value, momentum, and size between -2.0 and 2.0.
[SENTIMENT]: Identify actionable combinations of trailing 30-day sentiment score averages and news volume.
[INTEGRATION]: Formulate a SINGLE cross-domain boolean equation using AND/OR logic that contains both fundamental and sentiment variables in the same rule. NO point scoring."""
    
    for i in range(iterations):
        print(f"\n--- ACE Iteration {i+1} ---")
        
        fund_task = Task(
            description=f"Generate fundamental rules for {training_start} to {training_end}. Test hypotheses against 'ret_fwd_1'.\nCURRENT RULEBOOK:\n{current_rulebook}",
            expected_output=QUANT_EXPECTED_OUTPUT,
            agent=fundamentals_generator
        )
        
        sent_task = Task(
            description=f"Generate sentiment rules for {training_start} to {training_end}. Test hypotheses against 'ret_fwd_1'.\nCURRENT RULEBOOK:\n{current_rulebook}",
            expected_output=SENTIMENT_EXPECTED_OUTPUT, # <-- Updated here
            agent=sentiment_generator
        )
        
        reflect_task = Task(
            description="Analyze BOTH the fundamental and sentiment execution logs. Output targeted 'delta' updates.",
            expected_output="Structured list of delta updates for Fundamentals, Sentiment, and Signal Integration.",
            agent=reflector_agent,
            context=[fund_task, sent_task]
        )
        
        curate_task = Task(
            description=f"Integrate the Reflector's delta updates into the master rulebook.\nCURRENT RULEBOOK:\n{current_rulebook}",
            expected_output="The updated master rulebook.",
            agent=curator_agent
        )
        
        ace_crew = Crew(
            agents=[fundamentals_generator, sentiment_generator, reflector_agent, curator_agent],
            tasks=[fund_task, sent_task, reflect_task, curate_task],
            process=Process.sequential,
            verbose=False
        )
        current_rulebook = ace_crew.kickoff()
        
    return current_rulebook

def generate_zero_shot_signals(rulebook, eval_date):
    eval_task = Task(
        description=f"""Apply this optimized rulebook strictly to the datadate = '{eval_date}'.
        1. Explore extreme values for both fundamentals and 30-day sentiment at this date.
        2. Verify your rule combinations using both condition testing tools.
        3. Output the final strategy.
        
        OPTIMIZED RULEBOOK:
        {rulebook}
        """,
        expected_output=INTEGRATED_EXPECTED_OUTPUT,
        agent=evaluation_agent
    )
    
    eval_crew = Crew(
        agents=[evaluation_agent],
        tasks=[eval_task],
        process=Process.sequential,
        verbose=True
    )
    return eval_crew.kickoff()

# ============================================
# Main Execution (Monthly Loop)
# ============================================
if __name__ == "__main__":

    # FUNDAMENTALS_CSV_PATH = "../Data/green cleaned new.csv" 
    # RAVENPACK_CSV_PATH = "../Data/ravenpack_sp500_permno_final.csv"
    
    # try:
    #     setup_firm_database(FUNDAMENTALS_CSV_PATH)
    #     setup_sentiment_database(RAVENPACK_CSV_PATH)
    #     print("\nAll databases are ready! You can now run your main crewai script.")
    # except Exception as e:
    #     print(f"\nError creating databases: {e}")
    
    # Generate End-of-Month dates from April 2023 to Dec 2025
    # Note: Using 'ME' (MonthEnd) instead of 'M' which is deprecated in newer Pandas versions
    eval_dates = pd.date_range(start='2023-04-01', end='2023-05-31', freq='ME')
    
    for eval_dt in eval_dates:
        eval_date_str = eval_dt.strftime('%Y-%m-%d')
        
        print(f"\n{'='*70}\n EXECUTING PIPELINE FOR {eval_date_str}\n{'='*70}")
        PIPELINE_STATE["current_eval_date"] = eval_date_str
        
        # Define a trailing 12-month training window ending the month prior to evaluation
        train_end_dt = eval_dt.replace(day=1) - pd.Timedelta(days=1)
        train_start_dt = train_end_dt - pd.DateOffset(months=12)
        
        training_start = train_start_dt.strftime('%Y-%m-%d')
        training_end = train_end_dt.strftime('%Y-%m-%d')
        
        # PHASE 1: TRAINING (ACE Loop)
        PIPELINE_STATE["allow_returns"] = True
        print(f"Training ACE Rulebook from {training_start} to {training_end}...")
        optimized_rulebook = run_ace_loop(training_start, training_end, iterations=3)
        
        # PHASE 2: ZERO-SHOT EVAL
        PIPELINE_STATE["allow_returns"] = False
        print(f"\nApplying finalized rulebook zero-shot to {eval_date_str}...")
        final_strategy = generate_zero_shot_signals(optimized_rulebook, eval_date_str)
        
        # Save output using the evaluation month (e.g., systematic_strategy_04_2023_llama.txt)
        filename = f"systematic_strategy_{eval_dt.strftime('%m_%Y')}_llama.txt"
        with open(filename, 'w') as f:
            f.write(f"--- OPTIMIZED ACE RULEBOOK ---\n{optimized_rulebook}\n\n")
            f.write(f"--- FINAL ZERO-SHOT STRATEGY ---\n{final_strategy}")
            
        print(f"✓ Strategy saved to {filename}")


 EXECUTING PIPELINE FOR 2023-04-30
Training ACE Rulebook from 2022-03-31 to 2023-03-31...

--- ACE Iteration 1 ---


/var/folders/w9/w6pk180x2gz84c2y_110rdcw0000gn/T/ipykernel_18496/3707986721.py:586: DeprecationWarning: The 'generic' unit for NumPy timedelta is deprecated, and will raise an error in the future. This includes implicit conversion of bare integers (e.g. `+ 1`). Please use a specific unit instead.
  train_end_dt = eval_dt.replace(day=1) - pd.Timedelta(days=1)


╭─────────────────────────────────────────────── 🤖 Agent Started ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fundamental Quant Developer                                                                             │
│                                                                                                                 │
│  Task: Generate fundamental rules for 2022-03-31 to 2023-03-31. Test hypotheses against 'ret_fwd_1'.            │
│  CURRENT RULEBOOK:                                                                                              │
│  BASE RULEBOOK v1.0:                                                                                            │
│  [FUNDAMENTALS]: Find z-score combinations for value, momentum, and size between -2.0 and 2.0.                  │
│  [SENTIMENT]: Identify actionable combinations of trailing 30-day sentiment score averages and news volume.     │
│  [INTEGRATION]: Formulate a SINGLE cross-domain boolean equation using AND/OR logic that contains both          │
│  fundamental and sentiment variables in the same rule. NO point scoring.                                        │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────── 🔧 Agent Tool Execution ────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Fundamental Quant Developer                                                                             │
│                                                                                                                 │
│  Thought: Thought: I should analyze the provided data and develop explicit, systematic rules for generating     │
│  fundamental trading signals based on the market value of equity (mve), book-to-market ratio (bm), and          │
│  12-month momentum (mom12m).                                                                                    │
│                                                                                                                 │
│  Using Tool: get_database_schema                                                                                │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Input ───────────────────────────────────────────────────╮
│                                                                                                                 │
│  {}                                                                                                             │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭────────────────────────────────────────────────── Tool Output ──────────────────────────────────────────────────╮
│                                                                                                                 │
│  Database: firm_data.db                                                                                         │
│  Table: firms                                                                                                   │
│  Columns:                                                                                                       │
│    - PERMNO (INTEGER)                                                                                           │
│    - ticker (TEXT)                                                                                              │
│    - comnam (TEXT)                                                                                              │
│    - datadate (TEXT)                                                                                            │
│    - ret_fwd_1 (REAL)                                                                                           │
│    - mve_m (REAL)                                                                                               │
│    - mve (REAL)                                                                                                 │
│    - mom12m (REAL)                                                                                              │
│    - bm (REAL)                                                                                                  │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

... (237 outputs left out)

╭───────────────────────────────────────────── ✅ Agent Final Answer ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Agent: Final Strategy Evaluator                                                                                │
│                                                                                                                 │
│  Final Answer:                                                                                                  │
│  **FINAL INTEGRATED TRADING STRATEGY**                                                                          │
│                                                                                                                 │
│  1. FUNDAMENTAL RULES (mve, bm, mom12m as z-scores):                                                            │
│  - BUY if: mve > 1.5 AND mom12m > 0.7 AND news_count >= 5                                                       │
│  - SELL if: mve < -1.5 AND mom12m < -0.7 AND news_count <= 0                                                    │
│                                                                                                                 │
│  2. SENTIMENT RULES (avg_sentiment):                                                                            │
│  - BUY if: avg_sentiment > 0.85                                                                                 │
│  - SELL if: avg_sentiment < -0.85                                                                               │
│                                                                                                                 │
│  **COMBINED RULES**                                                                                             │
│                                                                                                                 │
│  - If both fundamental and sentiment rules are met, BUY                                                         │
│  - If neither fundamental nor sentiment rules are met, SELL                                                     │
│                                                                                                                 │
│  **FINAL ANSWER**                                                                                               │
│                                                                                                                 │
│  Based on the optimized rulebook, the final answer for the given date '2023-05-31' is: BUY                      │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 38b140d0-e326-402d-b59f-0204325f6b96                                                                     │
│  Agent: Final Strategy Evaluator                                                                                │
│  Tool Args:                                                                                                     │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: e61dc803-8eba-469e-867a-53856ae8abeb                                                                       │
│  Tool Args:                                                                                                     │
│  Final Output: **FINAL INTEGRATED TRADING STRATEGY**                                                            │
│                                                                                                                 │
│  1. FUNDAMENTAL RULES (mve, bm, mom12m as z-scores):                                                            │
│  - BUY if: mve > 1.5 AND mom12m > 0.7 AND news_count >= 5                                                       │
│  - SELL if: mve < -1.5 AND mom12m < -0.7 AND news_count <= 0                                                    │
│                                                                                                                 │
│  2. SENTIMENT RULES (avg_sentiment):                                                                            │
│  - BUY if: avg_sentiment > 0.85                                                                                 │
│  - SELL if: avg_sentiment < -0.85                                                                               │
│                                                                                                                 │
│  **COMBINED RULES**                                                                                             │
│                                                                                                                 │
│  - If both fundamental and sentiment rules are met, BUY                                                         │
│  - If neither fundamental nor sentiment rules are met, SELL                                                     │
│                                                                                                                 │
│  **FINAL ANSWER**                                                                                               │
│                                                                                                                 │
│  Based on the optimized rulebook, the final answer for the given date '2023-05-31' is: BUY                      │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

✓ Strategy saved to systematic_strategy_05_2023_llama.txt
